In [20]:
import pandas as pd
import folium
from folium import plugins
import numpy as np
from scipy.spatial.distance import cdist

import calliope

In [21]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the great-circle distance in kilometers between two points 
    on the earth (specified in decimal degrees).
    """
    # Convert decimal degrees to radians 
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    # Haversine formula 
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a)) 
    # Radius of earth in kilometers. Use 3956 for miles
    r = 6371 
    return c * r

In [22]:
# 1. Isolate the transmission network and coordinates
transmission_network = pd.read_csv('data_tables/transmission_network.csv')
coordinates_df = pd.read_csv('data_tables/manual_nodes_base_info.csv')

# 2. Merge coordinates onto the transmission network links
links_with_coords = pd.merge(
    transmission_network,
    coordinates_df[['nodes', 'latitude', 'longitude']],
    left_on='link_from',
    right_on='nodes',
    how='left'
).rename(columns={'latitude': 'lat_from', 'longitude': 'lon_from'})

links_with_coords = pd.merge(
    links_with_coords,
    coordinates_df[['nodes', 'latitude', 'longitude']],
    left_on='link_to',
    right_on='nodes',
    how='left',
    suffixes=('_from_node', '_to_node')
).rename(columns={'latitude': 'lat_to', 'longitude': 'lon_to'})

# 3. Interpolate points for each link and create the new links
interpolated_nodes_list = []
interpolated_links_list = []
num_interpolations = 10

for _, row in links_with_coords.iterrows():
    # --- This part generates the interpolated nodes (same as before) ---
    latitudes = np.linspace(row['lat_from'], row['lat_to'], num=num_interpolations + 2)
    longitudes = np.linspace(row['lon_from'], row['lon_to'], num=num_interpolations + 2)
    
    # Generate node names for the entire chain (start + interp + end)
    node_chain = (
        [row['link_from']] +
        [f"{row['link_from']}_{row['link_to']}_interp_{i+1}" for i in range(num_interpolations)] +
        [row['link_to']]
    )

    # Add the new interpolated nodes to a list for the new nodes DataFrame
    for i in range(num_interpolations):
        interpolated_nodes_list.append({
            'nodes': node_chain[i + 1],
            'latitude': latitudes[i + 1],
            'longitude': longitudes[i + 1],
            'comment': f"Interpolated node for link {row['techs']}"
        })

    # --- This new part generates the links between the nodes in the chain ---
    is_heat = row['techs'].startswith('TH')
    
    for i in range(len(node_chain) - 1):
        link_from_node = node_chain[i]
        link_to_node = node_chain[i+1]
        
        interpolated_links_list.append({
            'techs': f"{link_from_node}_to_{link_to_node}",
            'color': '#823739' if is_heat else '#6783E3',
            'name': 'Interpolated Heat transmission' if is_heat else 'Interpolated Electricity transmission',
            'base_tech': 'transmission',
            'flow_cap_max': '2000',
            'flow_out_eff_per_distance': '0.98' if is_heat else '0.99',
            'lifetime': '20',
            'link_from': link_from_node,
            'link_to': link_to_node
        })

# 4. Create the new DataFrames
new_nodes_df = pd.DataFrame(interpolated_nodes_list)
interpolated_links_df = pd.DataFrame(interpolated_links_list)

interpolated_links_df.to_csv('data_tables/interpolated_transmission_network.csv', index=False)

# 5. Append new nodes to the existing nodes_base_info.csv
# We use concat to combine the old and new data and then save, overwriting the old file.
nodes_base_info_df=pd.read_csv('data_tables/manual_nodes_base_info.csv')
updated_nodes_base_info_df = pd.concat([nodes_base_info_df, new_nodes_df], ignore_index=True)
updated_nodes_base_info_df.to_csv('data_tables/nodes_base_info.csv', index=False)

# 6. Update the nodes.csv file to define technologies at each interpolated node
existing_nodes_df = pd.read_csv('data_tables/manual_nodes.csv')

nodes_techs_to_add = []
for _, node_row in new_nodes_df.iterrows():
    node_name = node_row['nodes']
    
    # Determine if it's a heat or electricity node based on its name
    is_heat_path = 'TH' in node_name
    is_electricity_path = 'TE' in node_name
    
    # Add demand_heat or demand_electricity with sink_use_equals = 0
    # This makes the node capable of having demand for that carrier, even if it's zero.
    if is_heat_path:
        nodes_techs_to_add.append({
            'nodes': node_name,
            'techs': 'demand_heat',
            'parameters': 'sink_use_equals',
            'timesteps': '',
            '2050/01/01 00:00': 0
        })
    if is_electricity_path:
        nodes_techs_to_add.append({
            'nodes': node_name,
            'techs': 'demand_electricity',
            'parameters': 'sink_use_equals',
            'timesteps': '',
            '2050/01/01 00:00': 0
        })

new_nodes_techs_df = pd.DataFrame(nodes_techs_to_add)

# Concatenate with existing nodes_df and save
updated_nodes_df = pd.concat([existing_nodes_df, new_nodes_techs_df], ignore_index=True)
updated_nodes_df.to_csv('data_tables/nodes.csv', index=False)

# 5. Create Carrier DataFrames for the new links
interpolated_heat_links_techs = interpolated_links_df[interpolated_links_df['name'].str.contains('Heat')]['techs']
interpolated_electricity_links_techs = interpolated_links_df[interpolated_links_df['name'].str.contains('Electricity')]['techs']

interpolated_heat_carrier_df = pd.DataFrame({
    'techs': interpolated_heat_links_techs,
    'carrier_in': 1,
    'carrier_out': 1
})

interpolated_electricity_carrier_df = pd.DataFrame({
    'techs': interpolated_electricity_links_techs,
    'carrier_in': 1,
    'carrier_out': 1
})

# 6. Create Cost DataFrames for the new links
interpolated_heat_costs_df = pd.DataFrame({
    'techs': interpolated_heat_links_techs,
    'cost_flow_cap_per_distance': 100
})

interpolated_electricity_costs_df = pd.DataFrame({
    'techs': interpolated_electricity_links_techs,
    'cost_flow_cap_per_distance': 50
})

In [23]:
coordinates_df=pd.read_csv('data_tables/nodes_base_info.csv')
demand_nodes=coordinates_df[coordinates_df['nodes'].str.contains('D')].copy()
heat_transmission_nodes=coordinates_df[coordinates_df['nodes'].str.contains('TH')].copy()
electricity_transmission_nodes=coordinates_df[coordinates_df['nodes'].str.contains('TE')].copy()

# Extract coordinates for all three types
demand_coords = demand_nodes[['latitude', 'longitude']].values
heat_transmission_coords = heat_transmission_nodes[['latitude', 'longitude']].values
electricity_transmission_coords = electricity_transmission_nodes[['latitude', 'longitude']].values

# Calculate distances using the Haversine formula
# We pass the haversine function to cdist, which will apply it to every pair of coordinates.
heat_distances = cdist(demand_coords, heat_transmission_coords, 
                       lambda u, v: haversine_distance(u[0], u[1], v[0], v[1]))
electricity_distances = cdist(demand_coords, electricity_transmission_coords, 
                              lambda u, v: haversine_distance(u[0], u[1], v[0], v[1]))

# Find the index of the nearest node for each type separately
nearest_heat_nodes_idx = np.argmin(heat_distances, axis=1)
nearest_electricity_nodes_idx = np.argmin(electricity_distances, axis=1)

# Assign the correct nodes
demand_nodes['heat_node'] = heat_transmission_nodes.iloc[nearest_heat_nodes_idx]['nodes'].values
demand_nodes['electricity_node'] = electricity_transmission_nodes.iloc[nearest_electricity_nodes_idx]['nodes'].values

# Create links dataframe and write to csv
heat_links = pd.DataFrame({
    'techs': demand_nodes['nodes'] + '_to_' + demand_nodes['heat_node'],
    'color': '#823739',
    'name': 'Heat distribution',
    'base_tech': 'transmission',
    'flow_cap_max': '2000',
    'flow_out_eff_per_distance': '0.98',
    'lifetime': '20',
    'link_to': demand_nodes['nodes'],
    'link_from': demand_nodes['heat_node']
}).reset_index(drop=True)

electricity_links=pd.DataFrame({
    'techs': demand_nodes['nodes'] + '_to_' + demand_nodes['electricity_node'],
    'color': '#6783E3',
    'name': 'Electricity distribution',
    'base_tech': 'transmission',
    'flow_cap_max': '2000',
    'flow_out_eff_per_distance': '0.99',
    'lifetime': '20',
    'link_to': demand_nodes['nodes'],
    'link_from': demand_nodes['electricity_node']
}).reset_index(drop=True)

distribution_techs=pd.concat([heat_links, electricity_links], ignore_index=True)

transmission_network=pd.read_csv('data_tables/transmission_network.csv')
updated_links = pd.concat([interpolated_links_df, distribution_techs], ignore_index=True)
updated_links.to_csv('data_tables/links.csv', index=False)

# Create carrier dataframes and write to csv
distribution_heat=pd.DataFrame({
    'techs': demand_nodes['nodes'] + '_to_' +  demand_nodes['heat_node'],
    'carrier_out': '1',
    'carrier_in': '1'
    }).reset_index(drop=True)

transmission_heat=pd.read_csv('data_tables/transmission_heat.csv')
updated_heat_links=pd.concat([interpolated_heat_carrier_df, distribution_heat], ignore_index=True)
updated_heat_links.to_csv('data_tables/links_heat.csv', index=False)

distribution_electricity=pd.DataFrame({
    'techs': demand_nodes['nodes'] + '_to_' +  demand_nodes['electricity_node'],
    'carrier_out': '1',
    'carrier_in': '1'
    }).reset_index(drop=True)

transmission_electricity=pd.read_csv('data_tables/transmission_electricity.csv')
updated_electricity_links=pd.concat([interpolated_electricity_carrier_df, distribution_electricity], ignore_index=True)
updated_electricity_links.to_csv('data_tables/links_electricity.csv', index=False)

# Create costs dataframe and write to csv
distribution_heat_costs=pd.DataFrame({
    'techs': demand_nodes['nodes']  + '_to_' +  demand_nodes['heat_node'],
    'cost_flow_cap_per_distance': '100'
    }).reset_index(drop=True)

distribution_electricity_costs=pd.DataFrame({
    'techs': demand_nodes['nodes']  + '_to_' +  demand_nodes['electricity_node'],
    'cost_flow_cap_per_distance': '50'
    }).reset_index(drop=True)

distribution_costs=pd.concat([distribution_heat_costs, distribution_electricity_costs], ignore_index=True)

transmission_costs=pd.read_csv('data_tables/transmission_costs.csv')
updated_links_costs = pd.concat([interpolated_heat_costs_df, interpolated_electricity_costs_df, distribution_costs], ignore_index=True)
updated_links_costs.to_csv('data_tables/links_costs.csv', index=False)


In [24]:
calliope.set_log_verbosity("INFO", include_solver_output=True)

model = calliope.read_yaml("model.yaml")

[2025-11-12 16:09:44] INFO     Math init | loading pre-defined math.
[2025-11-12 16:09:44] INFO     Math init | loading math files {'spores', 'operate', 'storage_inter_cluster', 'base', 'milp'}.
[2025-11-12 16:09:44] INFO     Model: preprocessing data
[2025-11-12 16:09:44] INFO     Math build | building applied math with ['base'].
[2025-11-12 16:10:38] INFO     input data `color` not defined in model math; it will not be available in the optimisation problem.
[2025-11-12 16:10:38] INFO     input data `name` not defined in model math; it will not be available in the optimisation problem.
[2025-11-12 16:10:38] INFO     input data `comment` not defined in model math; it will not be available in the optimisation problem.
[2025-11-12 16:10:38] INFO     input data `link_from` not defined in model math; it will not be available in the optimisation problem.
[2025-11-12 16:10:38] INFO     input data `link_to` not defined in model math; it will not be available in the optimisation problem.
[2025

In [25]:
model.inputs

print(model.inputs.techs)

<xarray.DataArray 'techs' (techs: 2085)> Size: 17kB
array(['D100_to_TE53_TE52_interp_7', 'D100_to_TH53_TH52_interp_7',
       'D101_to_TE52_TE51_interp_4', ..., 'demand_heat', 'supply_electricity',
       'supply_geothermal'], dtype=object)
Coordinates:
  * techs    (techs) object 17kB 'D100_to_TE53_TE52_interp_7' ... 'supply_geo...


In [26]:
model.inputs.flow_cap_max.to_series().dropna()

techs
D100_to_TE53_TE52_interp_7               2000.0
D100_to_TH53_TH52_interp_7               2000.0
D101_to_TE52_TE51_interp_4               2000.0
D101_to_TH52_TH51_interp_4               2000.0
D102_to_TE52_TE51_interp_9               2000.0
                                          ...  
TH9_TH8_interp_8_to_TH9_TH8_interp_9     2000.0
TH9_TH8_interp_9_to_TH9_TH8_interp_10    2000.0
TH9_to_TH9_TH8_interp_1                  2000.0
supply_electricity                       2000.0
supply_geothermal                        2000.0
Name: flow_cap_max, Length: 2083, dtype: float64

In [27]:
model.inputs.sink_use_equals.sum(
    "timesteps", min_count=1, skipna=True
).to_series().dropna()

nodes             techs      
D1                demand_heat    10.0
D10               demand_heat    10.0
D100              demand_heat    10.0
D101              demand_heat    10.0
D102              demand_heat    10.0
                                 ... 
TH9_TH8_interp_5  demand_heat     0.0
TH9_TH8_interp_6  demand_heat     0.0
TH9_TH8_interp_7  demand_heat     0.0
TH9_TH8_interp_8  demand_heat     0.0
TH9_TH8_interp_9  demand_heat     0.0
Name: sink_use_equals, Length: 1921, dtype: float64

In [28]:
model.build()
model.solve()

[2025-11-12 16:10:43] INFO     Model: backend build starting
[2025-11-12 16:10:44] INFO     Optimisation Model | parameters/lookups | Generated.
[2025-11-12 16:10:46] INFO     Optimisation Model | variables | Generated.
[2025-11-12 16:11:08] INFO     Optimisation Model | global_expressions | Generated.
[2025-11-12 16:11:27] INFO     Optimisation Model | constraints | Generated.
[2025-11-12 16:11:27] INFO     Optimisation Model | piecewise_constraints | Generated.
[2025-11-12 16:11:28] INFO     Optimisation Model | objectives | Generated.
[2025-11-12 16:11:28] INFO     Model: backend build complete
[2025-11-12 16:11:28] INFO     Optimisation model | starting model in base mode.
[2025-11-12 16:11:29] DEBUG    Set parameter Username
Set parameter LicenseID to value 2716243
Academic license - for non-commercial use only - expires 2026-09-30
[2025-11-12 16:11:29] DEBUG    Read LP format model from file C:\Users\alexn\AppData\Local\Temp\tmpfwqi375g.pyomo.lp
Reading time = 0.05 seconds
x1: 20

In [29]:
model.results

<xarray.Dataset> Size: 611MB
Dimensions:                     (nodes: 1926, techs: 2085, carriers: 2,
                                 timesteps: 1, costs: 1)
Coordinates:
  * techs                       (techs) object 17kB 'D100_to_TE53_TE52_interp...
  * nodes                       (nodes) object 15kB 'D1' ... 'TH9_TH8_interp_9'
  * carriers                    (carriers) object 16B 'electricity' 'heat'
  * timesteps                   (timesteps) datetime64[ns] 8B 2050-01-01
  * costs                       (costs) object 8B 'monetary'
Data variables: (12/20)
    flow_cap                    (nodes, techs, carriers) float64 64MB nan ......
    link_flow_cap               (techs) float64 17kB 0.0 10.0 0.0 ... nan nan
    flow_out                    (nodes, techs, carriers, timesteps) float64 64MB ...
    flow_in                     (nodes, techs, carriers, timesteps) float64 64MB ...
    source_use                  (nodes, techs, timesteps) float64 32MB nan .....
    source_cap                  (nodes, techs) float64 32MB nan nan ... nan nan
    ...                          ...
    min_cost_optimisation       float64 8B 261.5
    capacity_factor             (nodes, techs, carriers, timesteps) float64 64MB ...
    systemwide_capacity_factor  (techs, carriers) float64 33kB 0.0 0.0 ... 1.0
    systemwide_levelised_cost   (techs, costs, carriers) float64 33kB nan ......
    total_levelised_cost        (costs, carriers) float64 16B 0.5223 0.1745
    unmet_sum                   (nodes, carriers, timesteps) float64 31kB 0.0...

In [30]:
df_heat = (
    model.results.flow_out.sel(carriers="heat")
    .sum("nodes", min_count=1, skipna=True)
    .to_series()
    .dropna()
    .unstack("techs")
)

df_heat.head()

techs,D100_to_TH53_TH52_interp_7,D101_to_TH52_TH51_interp_4,D102_to_TH52_TH51_interp_9,D103_to_TH51_TH50_interp_5,D104_to_TH48,D105_to_TH52_TH73_interp_10,D106_to_TH73_TH72_interp_4,D107_to_TH73_TH72_interp_9,D108_to_TH72_TH71_interp_4,D109_to_TH72_TH71_interp_9,...,TH9_TH8_interp_2_to_TH9_TH8_interp_3,TH9_TH8_interp_3_to_TH9_TH8_interp_4,TH9_TH8_interp_4_to_TH9_TH8_interp_5,TH9_TH8_interp_5_to_TH9_TH8_interp_6,TH9_TH8_interp_6_to_TH9_TH8_interp_7,TH9_TH8_interp_7_to_TH9_TH8_interp_8,TH9_TH8_interp_8_to_TH9_TH8_interp_9,TH9_TH8_interp_9_to_TH9_TH8_interp_10,TH9_to_TH9_TH8_interp_1,supply_geothermal
timesteps,,,,,,,,,,,,,,,,,,,,,
2050-01-01,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,...,1434.892689,1435.137925,1435.383202,1435.628521,1435.873882,1436.119285,1436.364729,1436.610215,1434.402344,1498.297884


In [31]:
df_electricity = (
    model.results.flow_out.sel(carriers="electricity")
    .sum("nodes", min_count=1, skipna=True)
    .to_series()
    .dropna()
    .unstack("techs")
)

df_electricity.head()

techs,D100_to_TE53_TE52_interp_7,D101_to_TE52_TE51_interp_4,D102_to_TE52_TE51_interp_9,D103_to_TE51_TE50_interp_5,D104_to_TE48,D105_to_TE52_TE73_interp_10,D106_to_TE73_TE72_interp_4,D107_to_TE73_TE72_interp_9,D108_to_TE72_TE71_interp_4,D109_to_TE72_TE71_interp_9,...,TE9_TE8_interp_2_to_TE9_TE8_interp_3,TE9_TE8_interp_3_to_TE9_TE8_interp_4,TE9_TE8_interp_4_to_TE9_TE8_interp_5,TE9_TE8_interp_5_to_TE9_TE8_interp_6,TE9_TE8_interp_6_to_TE9_TE8_interp_7,TE9_TE8_interp_7_to_TE9_TE8_interp_8,TE9_TE8_interp_8_to_TE9_TE8_interp_9,TE9_TE8_interp_9_to_TE9_TE8_interp_10,TE9_to_TE9_TE8_interp_1,supply_electricity
timesteps,,,,,,,,,,,,,,,,,,,,,
2050-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,500.645306


In [32]:
costs = model.results.cost.to_series().dropna()
costs.head()

nodes  techs                       costs   
D1     D1_to_TE30_SE5_interp_5     monetary    0.000000
       D1_to_TH30_TH29_interp_4    monetary    0.002781
D10    D10_to_TE20_TE19_interp_10  monetary    0.000000
       D10_to_TH20_TH19_interp_10  monetary    0.000473
D100   D100_to_TE53_TE52_interp_7  monetary    0.000000
Name: cost, dtype: float64

In [33]:
# We set the color mapping to use in all our plots by extracting the colors defined in the technology definitions of our model.
colors = model.inputs.color.to_series().to_dict()

df_electricity1 = (
    (model.results.flow_out.fillna(0) - model.results.flow_in.fillna(0))
    .sel(carriers="electricity")
    .sum("nodes")
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow in/out (kWh)")
    .reset_index()
)
df_electricity_demand = df_electricity1[df_electricity1.techs == "demand_electricity"]
df_electricity_other = df_electricity1[df_electricity1.techs != "demand_electricity"]

print(df_electricity1.head())


                                  techs  timesteps  Flow in/out (kWh)
0              TE1_SE1_interp_10_to_SE1 2050-01-01          -0.010739
1  TE1_SE1_interp_1_to_TE1_SE1_interp_2 2050-01-01          -0.010737
2  TE1_SE1_interp_2_to_TE1_SE1_interp_3 2050-01-01          -0.010737
3  TE1_SE1_interp_3_to_TE1_SE1_interp_4 2050-01-01          -0.010738
4  TE1_SE1_interp_4_to_TE1_SE1_interp_5 2050-01-01          -0.010738


In [34]:
carriers = ["heat", "electricity"]
df_flows = (
    (model.results.flow_out.fillna(0) - model.results.flow_in.fillna(0))
    .sel(carriers=carriers)
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow in/out (kWh)")
    .reset_index()
)
df_demand = df_flows[df_flows.techs.str.contains("demand")]
df_flows_other = df_flows[~df_flows.techs.str.contains("demand")]

print(df_flows.head())

  nodes                       techs carriers  timesteps  Flow in/out (kWh)
0    D1    D1_to_TH30_TH29_interp_4     heat 2050-01-01               10.0
1    D1                 demand_heat     heat 2050-01-01              -10.0
2   D10  D10_to_TH20_TH19_interp_10     heat 2050-01-01               10.0
3   D10                 demand_heat     heat 2050-01-01              -10.0
4  D100  D100_to_TH53_TH52_interp_7     heat 2050-01-01               10.0


In [35]:
df_capacity = (
    model.results.flow_cap.where(
        ~model.inputs.base_tech.str.contains("demand|transmission")
    )
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow capacity (kW)")
    .reset_index()
)

In [36]:
df_coords = model.inputs[["latitude", "longitude"]].to_dataframe().reset_index()
df_capacity = (
    model.results.flow_cap.where(model.inputs.base_tech == "transmission")
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow capacity (kW)")
    .reset_index()
)

# Define distribution and transmission dataframes for plotting
df_capacity_coords = pd.merge(df_coords, df_capacity, left_on="nodes", right_on="nodes").sort_values(by=['techs'])

# Extract link information from techs column (format: "node_from_to_node_to")
df_links = df_capacity_coords.copy()

# Split the techs column to get link_from and link_to
df_links[['link_from', 'link_to']] = df_links['techs'].str.rsplit('_to_', n=1, expand=True)

# Merge with df_coords twice to get both from and to coordinates
# First merge for "from" coordinates
df_links = df_links.merge(
    df_coords[['nodes', 'latitude', 'longitude']],
    left_on='link_from',
    right_on='nodes',
    how='left',
    suffixes=('', '_from')
)
df_links = df_links.rename(columns={'latitude': 'lat_from', 'longitude': 'lon_from'})

# Second merge for "to" coordinates
df_links = df_links.merge(
    df_coords[['nodes', 'latitude', 'longitude']],
    left_on='link_to',
    right_on='nodes',
    how='left',
    suffixes=('_temp', '_to')
)
df_links = df_links.rename(columns={'latitude': 'lat_to', 'longitude': 'lon_to'})

# Clean up duplicate columns
df_links = df_links.drop(columns=['nodes_temp', 'nodes_to'], errors='ignore')


# Create a Folium map centered on your data
center_lat = df_coords['latitude'].mean()
center_lon = df_coords['longitude'].mean()

map_fig = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=16,
    tiles='OpenStreetMap'
)


# Create FeatureGroups for different layers
demand_group = folium.FeatureGroup(name="Demand Nodes", show=True).add_to(map_fig)
supply_heat_group = folium.FeatureGroup(name="Supply Heat Nodes", show=True).add_to(map_fig)
supply_elec_group = folium.FeatureGroup(name="Supply Electricity Nodes", show=True).add_to(map_fig)
transmission_heat_group = folium.FeatureGroup(name="Heat Transmission Nodes", show=True).add_to(map_fig)
heat_link_group = folium.FeatureGroup(name="Heat Links", show=True).add_to(map_fig)
transmission_electricity_group = folium.FeatureGroup(name="Electricity Transmission Nodes", show=True).add_to(map_fig)
electricity_link_group = folium.FeatureGroup(name="Electricity Links", show=True).add_to(map_fig)

# Add lines for each link to the 'link_group' FeatureGroup
for idx, row in df_links.iterrows():
    if row['carriers'] == 'heat':
        color = 'green'
        target_group = heat_link_group
    else:
        color = 'blue'
        target_group = electricity_link_group
        
    folium.PolyLine(
        locations=[[row['lat_from'], row['lon_from']], [row['lat_to'], row['lon_to']]],
        color=color,
        weight=1,
        opacity=0.7,
        popup=f"<b>{row['techs']}</b><br>From: {row['link_from']}<br>To: {row['link_to']}<br>Capacity: {row['Flow capacity (kW)']} kW"
    ).add_to(target_group)


# Add node markers to their respective FeatureGroups
for idx, row in df_capacity_coords.iterrows():
    node_name = row['nodes']
    
    # Determine node type and styling
    if node_name.startswith('SH'):
        color = '#2ecc71' 
        radius = 1
        node_type = 'Supply heat'
        target_group = supply_heat_group
    elif node_name.startswith('SE'):
        color = "#2e38cc" 
        radius = 1
        node_type = 'Supply electricity'
        target_group = supply_elec_group
    elif node_name.startswith('D'):
        color = '#e74c3c'  
        radius = 1
        node_type = 'Demand'
        target_group = demand_group
    elif node_name.startswith('TH'):
        color = "#94d3ae" 
        radius = 1
        node_type = 'Transmission heat'
        target_group = transmission_heat_group
    else:  
        color = "#7076cc"  
        radius = 1
        node_type = 'Transmission electricity'
        target_group = transmission_electricity_group
    
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=radius,
        popup=f"<b>{row['nodes']}</b> ({node_type})<br>Capacity: {row['Flow capacity (kW)']} kW",
        color=color,
        fill=True,
        fillColor=color,
        fillOpacity=0.8,
        weight=2
    ).add_to(target_group) # Add to the correct group

# --- Add the LayerControl to the map ---
# This creates the toggle switch in the top-right corner
folium.LayerControl().add_to(map_fig)

# Display the map
map_fig.save("outputs/map_output.html")

In [ ]:
# For each item to be exported, find its name in inputs, merge with capacity data, and export to dataframe
tech_names = model.inputs.name.to_series().dropna()

total_flow_out = (
    model.results.flow_out
    .sum(dim=["nodes", "carriers", "timesteps"], min_count=1)
    .to_series()
    .dropna()
)

export_df = pd.DataFrame({
    'name': tech_names,
    'capacity_kw': total_flow_out
})

final_export_df = export_df[export_df['capacity_kw'] > 0].sort_values(by='name')

# For each item in the export dataframe, multiply capacity by some environmental impact factor, and add environmental impact column

environmental_impact_factors = {
    "National grid import": 1,  # e.g. gCO2/kW
    "Geothermal heat extraction": 1,   
    "Heat transmission": 1,             
    "Electricity transmission": 1,      
    "Heat distribution": 1,             
    "Electricity distribution": 1       
}

final_export_df['environmental_impact_per_kW'] = final_export_df['name'].map(environmental_impact_factors)

# Sum total environmental impact across all items and output total environmental impact

final_export_df['environmental_impact'] = final_export_df['capacity_kw'] * final_export_df['environmental_impact_per_kW']
final_export_df = final_export_df.reset_index()


total_impact=sum(final_export_df['environmental_impact'])

print(f"Total environmental impact: {total_impact} (unit)")

final_export_df.head()

techs
D100_to_TE53_TE52_interp_7     0.0
D100_to_TH53_TH52_interp_7    10.0
D101_to_TE52_TE51_interp_4     0.0
D101_to_TH52_TH51_interp_4    10.0
D102_to_TE52_TE51_interp_9     0.0
Name: flow_out, dtype: float64
Total environmental impact: nan (unit)


,techs,name,capacity_kw,environmental_impact_per_kW,environmental_impact
0,supply_geothermal,Geothermal heat extraction,1498.297884,1.0,1498.297884
1,D54_to_TH37_TH36_interp_4,Heat distribution,10.000000,1.0,10.000000
2,D55_to_TH36_TH33_interp_7,Heat distribution,10.000000,1.0,10.000000
3,D56_to_TH34_TH33_interp_3,Heat distribution,10.000000,1.0,10.000000
4,D57_to_TH34,Heat distribution,10.000000,1.0,10.000000
